# Credit Card Fraud Detection: Model Comparison and Evaluation

In this notebook, I will develop three models (Random Forest, XGBoost, Dense Neural Network) and compare them to choose the best option for credit card fraud detection.

The [dataset from Machine Learning Group - ULB](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), The dataset contains transactions made by credit cards in September 2013 by European cardholders. The dataset presents transactions that occurred in two days.

This dataset consists of the following features:
* time: Number of seconds elapsed between this transaction and the first transaction in the dataset.
* (V1 to V28): 28 PCA anonymous features.
* Amount: The transaction amount.
* Class: Binary label for fraud or not fraud.

# Data exploration

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("/kaggle/input/creditcardfraud/creditcard.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().values.any()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

From the cells above, we can gather some important information about the data:

1. All feature are already encoded as numerical values.
2. The dataset is large and contains no missing values, which makes it suitable for training models such as neural networks.
3. Our dataset had some duplicated values we cleaned for better performance.

The next important step is to check the balance between fraudulent and non-fraudulent transactions:

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

count_classes = df['Class'].value_counts().sort_index()

sns.barplot(x=count_classes.index, y=count_classes.values, palette=['green', 'red'])
plt.xticks(ticks=[0, 1], labels=["Non-fraud", "Fraud"])

plt.title("Transaction Class Distribution")
plt.xlabel("Class")
plt.ylabel("Frequency")
plt.show()

In [ ]:
frauds = df[df.Class == 1]
frauds.shape

In [ ]:
non_frauds = df[df.Class == 0]
non_frauds.shape

As we can expect the fraud cases are a rare event. Therefore the data is skewed.

Now I am going to check if there is any insight on the amount and the transaction type. Besides, I am going to see if there is any correlation on time and transactions.

In [ ]:
frauds.Amount.describe()

In [ ]:
non_frauds.Amount.describe()

In [ ]:
sns.set(style="whitegrid")

f, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
f.suptitle('Amount per transaction by class')


sns.histplot(data=frauds.Amount, bins=40, ax=ax1, color='red')
ax1.set_title('Fraud')
ax1.set_yscale('log')

sns.histplot(data=non_frauds.Amount, bins=40, ax=ax2, color='green')
ax2.set_title('Normal')

plt.xlabel('Amount ($)')
plt.ylabel('Number of Transactions')
plt.xlim((0, 20000))
plt.yscale('log')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
sns.set(style="whitegrid")

f, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 8))
f.suptitle('Time of transaction vs Amount by class', fontsize=16)

sns.scatterplot(
    data=frauds,
    x='Time',
    y='Amount',
    ax=ax1,
    color='red',
    alpha=0.6,
    s=10
)
ax1.set_title('Fraud', fontsize=14)
ax1.set_ylabel('Amount')
ax1.set_xlabel('')

sns.scatterplot(
    data=non_frauds,
    x='Time',
    y='Amount',
    ax=ax2,
    color='green',
    alpha=0.2,
    s=10
)
ax2.set_title('Non-Fraud', fontsize=14)
ax2.set_xlabel('Time (in Seconds)')
ax2.set_ylabel('Amount')

plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

The time feature does not seem to be relevant for distinguish fraud or non-fraud. So I will remove it to keep useful features on the training.

Furthermore I am going to scale the Amount column.

In [ ]:
from sklearn.preprocessing import StandardScaler

df = df.drop(['Time'], axis=1)
df['Amount'] = StandardScaler().fit_transform(df['Amount'].values.reshape(-1, 1))

Finally, I am going to check if there is any correlation between the features of the dataset using a heatmap.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), cmap='coolwarm', center=0, annot=False)

The correlation heatmap reveals some dependencies among certain PCA components (e.g., V7–Amount and V20-Amount), and highlights which features are most predictive of fraudulent transactions.

# Building the models
Now, with those insights we can start developing our models to detect fraud.

We are going to build three models:
1. Random Forest Classifier
2. XGBoost
3. Dense Neural Network

# Data preparation

In [ ]:
from sklearn.preprocessing import StandardScaler

df['Amount'] = StandardScaler().fit_transform(df['Amount'].values.reshape(-1, 1))

X = df.drop(['Class'], axis=1) # All features except the target
y = df['Class'] # The target variable

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("Training Random Forest Classifier...")
rf_model.fit(X_train, y_train)
print("Random Forest Training Complete.")

# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("\n--- Random Forest Classifier Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1-Score:", f1_score(y_test, y_pred_rf))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

# XGBoost

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(objective='binary:logistic',
                              n_estimators=100,
                              learning_rate=0.1,
                              use_label_encoder=False,
                              eval_metric='logloss',
                              random_state=42,
                              n_jobs=-1)

# Train the model
print("\nTraining XGBoost Classifier...")
xgb_model.fit(X_train, y_train)
print("XGBoost Training Complete.")

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("\n--- XGBoost Classifier Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1-Score:", f1_score(y_test, y_pred_xgb))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))

# Dense Neural Network

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

# Determine the number of features (input dimension)
input_dim = X_train.shape[1]

# Build the Sequential model
nn_model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
nn_model.compile(optimizer='adam',
                 loss='binary_crossentropy',
                 metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()])

# Train the model
print("\nTraining Dense Neural Network...")
history = nn_model.fit(X_train, y_train,
                       epochs=30,
                       batch_size=32,
                       validation_split=0.1,
                       verbose=1)
print("Neural Network Training Complete.")

# Make predictions on the test set
y_proba_nn = nn_model.predict(X_test).flatten()
y_pred_nn = (y_proba_nn > 0.5).astype(int)

# Evaluate the model
print("\n--- Dense Neural Network Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Precision:", precision_score(y_test, y_pred_nn))
print("Recall:", recall_score(y_test, y_pred_nn))
print("F1-Score:", f1_score(y_test, y_pred_nn))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_nn))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nn))

# Conclusion

| Metric        | Random Forest     | XGBoost           | Dense Neural Network |
|---------------|:-------------------:|:-------------------:|:----------------------:|
| Accuracy      | 0.99952           | 0.99949           | 0.99942              |
| Precision     | 0.97222           | 0.94595           | 0.89744              |
| Recall        | 0.73684           | 0.73684           | 0.73684              |
| F1-Score      | 0.83832           | 0.82840           | 0.80925              |
| ROC AUC Score | 0.92938           | 0.97486           | 0.96586              |


Based on the metrics, the **Random Forest Classifier** is the best model overall.

While all models achieve similar accuracy due to class imbalance, Random Forest stands out with the highest **F1-Score (0.838)** and **Precision (0.972)** for the minority class, indicating better balance between detecting fraud (recall) and minimizing false positives.

Even though XGBoost has a slightly higher **ROC AUC (0.975)**, Random Forest's stronger performance in key metrics like F1 and precision makes it the most reliable choice for this fraud detection task.
